In [15]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, GroupKFold
from sklearn.metrics import make_scorer, mean_squared_error, r2_score, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("final_data.csv")

target_col = "Learning.Adjusted.Years.of.School"
groups = df["Country.x"]
X = df.drop(columns=[target_col, "Country.x", "key", "Currency"], errors="ignore")
y = df[target_col]

feature_cols = X.select_dtypes(include=["number", "integer"]).columns.drop(["Year"], errors="ignore")

preprocessor = ColumnTransformer(transformers=[
    ("impute_and_scale", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]), feature_cols)
], remainder="passthrough")

scoring = {
    "MAE": make_scorer(mean_absolute_error),
    "MSE": make_scorer(mean_squared_error),
    "R2": make_scorer(r2_score)
}

gkf = GroupKFold(n_splits=5)

models = {
    "RandomForest": RandomForestRegressor(random_state=42),
    "LinearRegression": LinearRegression(),
    "BayesianRidge": BayesianRidge(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "Bagging": BaggingRegressor(random_state=42),
    "Voting": VotingRegressor([
        ("lr", LinearRegression()),
        ("rf", RandomForestRegressor(random_state=42)),
        ("knn", KNeighborsRegressor()),
        ("svr", SVR()),
        ("br", BayesianRidge()),
        ("ada", AdaBoostRegressor(random_state=42)),
        ("bag", BaggingRegressor(random_state=42))
    ]),
    "Stacking": StackingRegressor(
        estimators=[
            ("lr", LinearRegression()),
            ("rf", RandomForestRegressor(random_state=42)),
            ("knn", KNeighborsRegressor()),
            ("svr", SVR())
        ],
        final_estimator=BayesianRidge()
    )
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("regressor", model)
    ])

    cv = cross_validate(
        pipeline, X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring=scoring
    )

    results.append({
        "Model": name,
        "MAE": np.mean(cv["test_MAE"]),
        "MSE": np.mean(cv["test_MSE"]),
        "R2": np.mean(cv["test_R2"])
    })

results_df = pd.DataFrame(results)
results_df["MAE_rank"] = results_df["MAE"].rank()
results_df["MSE_rank"] = results_df["MSE"].rank()
results_df["R2_rank"] = results_df["R2"].rank(ascending=False)

results_df["Avg_Rank"] = results_df[["MAE_rank", "MSE_rank", "R2_rank"]].mean(axis=1)
results_df = results_df.sort_values(by="Avg_Rank")

print(results_df[["Model", "MAE", "MSE", "R2", "Avg_Rank"]])

              Model       MAE       MSE        R2  Avg_Rank
7            Voting  0.924882  1.417214  0.792227  1.000000
8          Stacking  0.931994  1.459441  0.790846  2.000000
1  LinearRegression  0.951152  1.631013  0.773909  3.666667
2     BayesianRidge  0.964396  1.611385  0.773871  4.333333
5          AdaBoost  0.960420  1.613615  0.760615  4.666667
0      RandomForest  0.958588  1.703786  0.746550  6.000000
6           Bagging  0.970196  1.691349  0.750170  6.333333
3               KNN  1.095097  2.118554  0.684663  8.000000
4               SVR  2.285704  7.392400 -0.040514  9.000000
